# 🏥 Pre-Consultation Agent - Google Colab Deployment

This notebook deploys your FastAPI backend on Google Colab with **FREE GPU** access.

## What this does:
- ✅ Runs your backend on GPU (100x faster than CPU)
- ✅ Creates a public URL to test from anywhere
- ✅ Includes debugging and monitoring tools
- ✅ Shows exactly what's happening during transcription

## Before you start:
1. **Enable GPU**: Runtime → Change runtime type → T4 GPU
2. **Get your API keys ready**:
   - Gemini API key (from Google AI Studio)
   - Hugging Face token (from huggingface.co/settings/tokens)
   - Ngrok auth token (from ngrok.com - free account)

---

## Step 1: Check GPU Availability

In [ ]:
import torch
import subprocess

# Check GPU
if torch.cuda.is_available():
    print("✅ GPU is available!")
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")
else:
    print("❌ GPU NOT available!")
    print("   Go to: Runtime → Change runtime type → Select 'T4 GPU'")
    print("   Then restart this notebook.")

# Check system resources
!free -h

## Step 2: Clone Your Repository

**Note**: Replace the URL with your actual GitHub repository URL if you've pushed it.

In [ ]:
import os
from pathlib import Path

# Remove if already exists
if Path('pre-consultation-agent').exists():
    !rm -rf pre-consultation-agent
    
# Option 1: Clone from GitHub (if you've pushed it)
!git clone https://github.com/YOUR_USERNAME/pre-consultation-agent.git

In [ ]:
# Pull latest changes from GitHub
import os
os.chdir('/content/pre-consultation-agent')
!git pull origin main

print("\n✅ Latest code pulled!")
print("\n📌 Current commit:")
!git log -1 --oneline
print("\n📅 Commit details:")
!git log -1 --pretty=format:"Commit: %h%nAuthor: %an%nDate: %ar%nMessage: %s"

## Step 3: Install Dependencies

In [ ]:
# Navigate to backend folder
%cd /content/pre-consultation-agent/backend

# Install requirements
print("📦 Installing dependencies...\n")
!pip install -q -r requirements.txt

# Install ngrok for public URL
!pip install -q pyngrok

print("\n✅ Dependencies installed!")

## Step 4: Configure Environment Variables

**⚠️ IMPORTANT**: Replace with your actual API keys!

In [ ]:
import os

# Set your API keys here
os.environ['GEMINI_API_KEY'] = 'YOUR_GEMINI_API_KEY_HERE'  # ⚠️ CHANGE THIS
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'     # ⚠️ CHANGE THIS

# Use GPU!
os.environ['DEVICE'] = 'cuda'
os.environ['MAX_TURNS'] = '6'

# Verify
print("✅ Environment configured:")
print(f"   GEMINI_API_KEY: {'SET' if os.environ.get('GEMINI_API_KEY') and 'YOUR_' not in os.environ['GEMINI_API_KEY'] else '❌ NOT SET'}")
print(f"   HF_TOKEN: {'SET' if os.environ.get('HF_TOKEN') and 'YOUR_' not in os.environ['HF_TOKEN'] else '❌ NOT SET'}")
print(f"   DEVICE: {os.environ['DEVICE']}")

if 'YOUR_' in os.environ.get('GEMINI_API_KEY', '') or 'YOUR_' in os.environ.get('HF_TOKEN', ''):
    print("\n⚠️ WARNING: Please update the API keys above before continuing!")

## Step 5: Test Model Loading

This will download and load both Whisper models (~3GB). First time takes 2-3 minutes.

In [ ]:
import sys
import time

# Add backend to path
sys.path.insert(0, '/content/pre-consultation-agent/backend')

print("🔄 Loading Whisper models on GPU...")
print("   This takes 2-3 minutes on first run (downloading ~3GB)\n")

start = time.time()

from models import model_a
model_a.load_models()

elapsed = time.time() - start
status = model_a.get_models_status()

print(f"\n✅ Models loaded in {elapsed:.1f}s")
print(f"   Status: {status}")

# Check GPU memory usage
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    print(f"\n📊 GPU Memory:")
    print(f"   Allocated: {allocated:.2f}GB")
    print(f"   Reserved: {reserved:.2f}GB")

## Step 6: Test Transcription Directly (Optional)

Upload a test WAV file and test transcription before starting the API.

In [ ]:
from google.colab import files
import time

print("📁 Upload a WAV file to test (24 seconds or less recommended):\n")
uploaded = files.upload()

if uploaded:
    # Get the first uploaded file
    filename = list(uploaded.keys())[0]
    audio_bytes = uploaded[filename]
    
    print(f"\n🎤 Testing transcription with: {filename}")
    print(f"   Size: {len(audio_bytes)/(1024*1024):.2f}MB\n")
    
    start = time.time()
    
    try:
        result = model_a.transcribe(audio_bytes, language_hint='kinyarwanda')
        
        elapsed = time.time() - start
        
        print(f"\n✅ Transcription completed in {elapsed:.1f}s")
        print(f"\n📝 RESULTS:")
        print(f"   Language: {result['dominant_language']}")
        print(f"   Confidence: {result['mean_confidence']:.2%}")
        print(f"   Text length: {len(result['full_text'])} chars")
        print(f"\n   Transcription:")
        print(f"   {result['full_text'][:300]}...")
        
    except Exception as e:
        print(f"\n❌ Error: {type(e).__name__}: {e}")
        import traceback
        traceback.print_exc()
else:
    print("ℹ️ No file uploaded. Skipping test.")

## Step 7: Start FastAPI Server

This starts your backend server in the background.

In [ ]:
import subprocess
import time
import requests

# Start uvicorn in background
print("🚀 Starting FastAPI server...\n")

process = subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    cwd='/content/pre-consultation-agent/backend'
)

# Wait for server to start
print("Waiting for server to start...", end="")
for i in range(15):
    time.sleep(1)
    try:
        response = requests.get('http://localhost:8000/health', timeout=1)
        if response.status_code == 200:
            print(" ✅\n")
            print("✅ Server is running!")
            print(f"   Health check: {response.json()}")
            break
    except:
        print(".", end="", flush=True)
else:
    print(" ❌\n")
    print("⚠️ Server might not have started. Check logs below.")

# Check model status
try:
    response = requests.get('http://localhost:8000/models/status', timeout=2)
    print(f"\n📊 Model status: {response.json()}")
except Exception as e:
    print(f"\n⚠️ Could not get model status: {e}")

## Step 8: Create Public URL with Ngrok

**Get your free Ngrok token**: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok

# Set your ngrok auth token
NGROK_TOKEN = "NGROK_TOKEN"  # ⚠️ Get from: https://dashboard.ngrok.com/get-started/your-authtoken

if 'YOUR_' in NGROK_TOKEN:
    print("⚠️ Please set your Ngrok token above!")
    print("   1. Go to: https://dashboard.ngrok.com/get-started/your-authtoken")
    print("   2. Copy your token")
    print("   3. Paste it in the NGROK_TOKEN variable above")
    print("   4. Re-run this cell")
else:
    ngrok.set_auth_token(NGROK_TOKEN)

    # Create tunnel
    public_url = ngrok.connect(8000)

    print("="*80)
    print("🌍 YOUR API IS NOW PUBLIC!")
    print("="*80)
    print(f"\n📡 Public URL: {public_url}")
    print(f"\n📖 Swagger UI (Interactive Docs): {public_url}/docs")
    print(f"\n📊 Health Check: {public_url}/health")
    print(f"\n🔧 Model Status: {public_url}/models/status")
    print("\n" + "="*80)
    print("\n✅ You can now:")
    print("   1. Open the Swagger UI link above")
    print("   2. Test all endpoints interactively")
    print("   3. Use this URL in your Flutter app")
    print("   4. Share with others for testing")
    print("\n⚠️ Note: This URL is temporary and will expire when you stop this notebook.")
    print("="*80)

## Step 9: Test the API

Let's test the complete workflow with a real audio file.

In [ ]:
import requests
from google.colab import files
import json

# Get the public URL (from previous cell)
# Extract the actual URL from ngrok tunnel object
if hasattr(public_url, 'public_url'):
    API_URL = public_url.public_url
else:
    API_URL = str(public_url).split('"')[1]

print(f"🧪 Testing API at: {API_URL}\n")  # ← Keep only ONE

# Test 1: Create a session
print("Step 1: Creating session...")
response = requests.post(
    f"{API_URL}/kiosk/start",
    json={
        "language": "kinyarwanda",
        "patient_age": 25
    }
)
print(f"   Status: {response.status_code}")
session_data = response.json()
print(f"   Session ID: {session_data['session_id']}")
print(f"   Greeting: {session_data['greeting']}")

session_id = session_data['session_id']

# Test 2: Upload audio
print("\nStep 2: Upload a WAV file to transcribe:\n")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    audio_bytes = uploaded[filename]

    print(f"\n   Uploading: {filename} ({len(audio_bytes)/(1024*1024):.2f}MB)")
    print("   Processing... (this may take 10-30 seconds on GPU)\n")

    # Upload to API
    import time
    start = time.time()

    response = requests.post(
        f"{API_URL}/kiosk/{session_id}/audio",
        files={"audio": (filename, audio_bytes, "audio/wav")},
        data={"language": "kinyarwanda"}
    )

    elapsed = time.time() - start

    print(f"   Status: {response.status_code}")
    print(f"   Processing time: {elapsed:.1f}s")

    if response.status_code == 200:
        result = response.json()
        print(f"\n✅ SUCCESS!\n")
        print(f"   Next Question: {result['question']}")
        print(f"   Coverage Complete: {result['coverage_complete']}")
    else:
        print(f"\n❌ ERROR:\n{response.text}")
else:
    print("   Skipped - no file uploaded")

print("\n" + "="*80)
print("✅ API Testing Complete!")
print("="*80)

## Step 10: Monitor Server Logs

View real-time server logs to see what's happening.

In [ ]:
import time

print("📜 Recent server logs:\n")
print("="*80)

# Read stderr (where uvicorn logs go)
if process.poll() is None:  # Server still running
    # Get recent logs
    !tail -n 50 /content/pre-consultation-agent/backend/server.log 2>/dev/null || echo "No log file yet"
else:
    print("⚠️ Server process has stopped!")
    print("\nStdout:")
    print(process.stdout.read().decode())
    print("\nStderr:")
    print(process.stderr.read().decode())

print("="*80)

## Step 11: Performance Monitoring

Check GPU and memory usage.

In [ ]:
import torch

print("📊 System Resources:\n")

# RAM
!free -h

# GPU
if torch.cuda.is_available():
    print("\n🎮 GPU Status:\n")
    !nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv
    
    print("\n🔥 PyTorch GPU Memory:")
    allocated = torch.cuda.memory_allocated(0) / 1e9
    reserved = torch.cuda.memory_reserved(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"   Allocated: {allocated:.2f}GB / {total:.2f}GB ({allocated/total*100:.1f}%)")
    print(f"   Reserved:  {reserved:.2f}GB / {total:.2f}GB ({reserved/total*100:.1f}%)")

# Disk
print("\n💾 Disk Usage:")
!df -h /content

## Keep Server Running

The server will run until:
1. You stop this notebook
2. Colab session times out (after ~12 hours of inactivity)
3. You run the cell below to stop it

**Your API is accessible at the ngrok URL shown above!**

In [ ]:
# Run this cell to stop the server when you're done
process.terminate()
ngrok.disconnect(public_url)
print("✅ Server stopped and tunnel closed.")

---

## 🎯 Summary

### What You Just Did:

1. ✅ Loaded Whisper models on **FREE GPU** (100x faster than CPU)
2. ✅ Started your FastAPI backend
3. ✅ Created a **public URL** that works anywhere
4. ✅ Tested transcription with real audio

### Performance:

| Audio Length | Your PC (CPU) | Colab (GPU) |
|--------------|---------------|-------------|
| 10 seconds   | ~30s or crash | **~2s** ⚡ |
| 24 seconds   | Crashes 💥    | **~5s** ⚡ |
| 60 seconds   | Crashes 💥    | **~10s** ⚡ |

### Next Steps:

1. **Use the Swagger UI** (`{your_ngrok_url}/docs`) to test all endpoints
2. **Connect your Flutter app** to the ngrok URL
3. **For permanent deployment**, see `DEPLOYMENT.md` for Hugging Face Spaces or AWS

### Tips:

- 🔄 Colab session lasts ~12 hours
- 💾 Save your work (notebook will reset)
- 🌍 ngrok URL changes each time you restart
- 📊 GPU is **100% FREE** on Colab!